# Holt's Linear Trend Method

Simple Exponential Smoothing produces flat forecasts because it has only a
level component.  **Holt's method** (1957) extends SES by adding a
**trend component**, allowing the forecast to increase or decrease over time.

We also cover the **damped trend** modification (Gardner & McKenzie, 1985),
which prevents the forecast from extrapolating the trend indefinitely — one
of the most successful modifications in the history of forecasting.

**Key references:**
- FPP3 Section 8.2
- Portilla TSA Section 05: Double Exponential Smoothing

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.holtwinters import Holt
from sklearn.metrics import mean_absolute_error, mean_squared_error

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["lines.linewidth"] = 1.5

DATA_DIR = Path("../data")

## 1. Load Data

We use two datasets:
- **Airline passengers**: strong trend + seasonality (to show why Holt alone
  is not enough when seasonality is present, but good for trend capture).
- **US population**: pure trend, no seasonality — ideal for Holt's method.

In [ ]:
airline = pd.read_csv(
    DATA_DIR / "airline_passengers.csv",
    index_col="Month",
    parse_dates=True,
)
airline.index.freq = "MS"
airline.columns = ["Passengers"]

print(f"Airline passengers: {airline.shape}")
airline.head()

In [ ]:
uspop = pd.read_csv(
    DATA_DIR / "uspopulation.csv",
    index_col="DATE",
    parse_dates=True,
)
uspop.index.freq = "MS"
uspop.columns = ["Population"]

print(f"US population: {uspop.shape}")
print(f"Date range: {uspop.index[0].date()} to {uspop.index[-1].date()}")
uspop.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(airline["Passengers"])
axes[0].set_title("Airline Passengers (trend + seasonality)")
axes[0].set_ylabel("Thousands of Passengers")

axes[1].plot(uspop["Population"])
axes[1].set_title("US Population Estimates (trend, no seasonality)")
axes[1].set_ylabel("Population (thousands)")

plt.tight_layout()
plt.show()

## 2. Train / Test Split

In [ ]:
# Airline: first 120 months train, last 24 test
air_train = airline.iloc[:120]
air_test = airline.iloc[120:]

# US population: first 72 months train, last ~23 test
pop_train = uspop.iloc[:72]
pop_test = uspop.iloc[72:]

print(f"Airline — Train: {len(air_train)}, Test: {len(air_test)}")
print(f"US Pop  — Train: {len(pop_train)}, Test: {len(pop_test)}")

---
## 3. Holt's Linear Trend — Component Form

Holt's method uses two smoothing equations and a forecast equation:

$$
\text{Forecast:} \quad \hat{y}_{t+h|t} = \ell_t + h b_t
$$

$$
\text{Level:} \quad \ell_t = \alpha y_t + (1-\alpha)(\ell_{t-1} + b_{t-1})
$$

$$
\text{Trend:} \quad b_t = \beta^*(\ell_t - \ell_{t-1}) + (1-\beta^*)b_{t-1}
$$

**Two parameters:**
- $\alpha$ — controls smoothing of the **level** (same role as in SES)
- $\beta^*$ — controls smoothing of the **trend** (slope)

The forecast is no longer flat — it is a **straight line** with slope $b_t$,
starting from level $\ell_t$.

In [ ]:
def holt_from_scratch(y, alpha, beta, initial_level=None, initial_trend=None):
    """Holt's linear trend method implemented from scratch."""
    y = np.asarray(y, dtype=float)
    n = len(y)
    levels = np.zeros(n)
    trends = np.zeros(n)

    # Initialize
    levels[0] = initial_level if initial_level is not None else y[0]
    trends[0] = initial_trend if initial_trend is not None else (y[1] - y[0])

    for t in range(1, n):
        levels[t] = alpha * y[t] + (1 - alpha) * (levels[t-1] + trends[t-1])
        trends[t] = beta * (levels[t] - levels[t-1]) + (1 - beta) * trends[t-1]

    return levels, trends


# Apply to US population
y_pop = pop_train["Population"].values
levels, trends = holt_from_scratch(y_pop, alpha=0.8, beta=0.2)

print(f"Last level: {levels[-1]:.0f}")
print(f"Last trend: {trends[-1]:.1f} per month")

In [ ]:
# Visualize level and trend components
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(pop_train.index, y_pop, color="black", alpha=0.5, label="Actual")
axes[0].plot(pop_train.index, levels, color="tab:blue", label="Level")
axes[0].set_title("Holt's Method — Level Component")
axes[0].set_ylabel("Population")
axes[0].legend()

axes[1].plot(pop_train.index, trends, color="tab:red")
axes[1].set_title("Holt's Method — Trend Component (slope per period)")
axes[1].set_ylabel("Trend (change per month)")

plt.tight_layout()
plt.show()

---
## 4. statsmodels Implementation — Holt's Method

In [ ]:
# Holt's linear trend on US population
model_holt = Holt(
    pop_train["Population"],
    initialization_method="estimated",
)
fit_holt = model_holt.fit(optimized=True)

print(f"Alpha (level):  {fit_holt.params['smoothing_level']:.4f}")
print(f"Beta* (trend):  {fit_holt.params['smoothing_trend']:.4f}")
print(f"Initial level:  {fit_holt.params['initial_level']:.2f}")
print(f"Initial trend:  {fit_holt.params['initial_trend']:.4f}")
print(f"AIC:            {fit_holt.aic:.2f}")

In [ ]:
fit_holt.summary()

In [ ]:
# Forecast
fc_holt = fit_holt.forecast(len(pop_test))

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(pop_train["Population"], label="Train", color="black", alpha=0.5)
ax.plot(pop_test["Population"], label="Actual (test)", color="tab:blue", linewidth=2)
ax.plot(fc_holt, label="Holt's linear", color="tab:red", linestyle="--", linewidth=2)
ax.set_title("Holt's Linear Trend — US Population")
ax.set_ylabel("Population (thousands)")
ax.legend()
plt.tight_layout()
plt.show()

mae_holt = mean_absolute_error(pop_test["Population"], fc_holt)
print(f"MAE (Holt's linear): {mae_holt:.2f}")

---
## 5. Damped Trend — Gardner & McKenzie (1985)

Holt's linear trend extrapolates the trend **indefinitely**, which is often
unrealistic for long forecast horizons.  The **damped trend** adds a
parameter $\phi$ ($0 < \phi < 1$) that causes the trend to **flatten**
over time:

$$
\hat{y}_{t+h|t} = \ell_t + (\phi + \phi^2 + \cdots + \phi^h) b_t
$$

$$
\ell_t = \alpha y_t + (1-\alpha)(\ell_{t-1} + \phi b_{t-1})
$$

$$
b_t = \beta^*(\ell_t - \ell_{t-1}) + (1-\beta^*) \phi b_{t-1}
$$

Key properties:
- When $\phi = 1$: this reduces to standard Holt's (no damping).
- When $\phi < 1$: forecasts converge to $\ell_T + \frac{\phi}{1-\phi} b_T$
  as $h \to \infty$.
- Typical range: $\phi \in [0.80, 0.98]$.
- **Empirically, damped trend methods are among the most accurate** across
  large-scale forecasting competitions (M3, M4).

In [ ]:
# Visualize how phi affects the forecast trajectory
fig, ax = plt.subplots(figsize=(10, 5))

h = np.arange(1, 51)
level_T = 100
trend_T = 2

for phi in [1.0, 0.98, 0.95, 0.90, 0.80]:
    if phi == 1.0:
        fc = level_T + h * trend_T
        label = f"phi = {phi:.2f} (linear)"
    else:
        phi_sum = np.array([sum(phi**j for j in range(1, hi+1)) for hi in h])
        fc = level_T + phi_sum * trend_T
        asymptote = level_T + phi / (1 - phi) * trend_T
        label = f"phi = {phi:.2f} (limit = {asymptote:.1f})"
    ax.plot(h, fc, label=label)

ax.set_title("Effect of Damping Parameter phi on Forecast Trajectory")
ax.set_xlabel("Forecast horizon h")
ax.set_ylabel("Forecast value")
ax.legend()
plt.tight_layout()
plt.show()

print("phi = 1.0: linear trend continues forever (standard Holt).")
print("phi < 1.0: trend flattens — smaller phi means faster damping.")

In [ ]:
# Fit damped trend model
model_damped = Holt(
    pop_train["Population"],
    damped_trend=True,
    initialization_method="estimated",
)
fit_damped = model_damped.fit(optimized=True)

print(f"Alpha (level):  {fit_damped.params['smoothing_level']:.4f}")
print(f"Beta* (trend):  {fit_damped.params['smoothing_trend']:.4f}")
print(f"Phi (damping):  {fit_damped.params['damping_trend']:.4f}")
print(f"AIC:            {fit_damped.aic:.2f}")

In [ ]:
fit_damped.summary()

---
## 6. Linear Trend vs Damped Trend — Comparison

In [ ]:
fc_damped = fit_damped.forecast(len(pop_test))

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(pop_train["Population"], label="Train", color="black", alpha=0.5)
ax.plot(pop_test["Population"], label="Actual (test)", color="tab:blue", linewidth=2)
ax.plot(fc_holt, label="Holt linear", color="tab:red", linestyle="--", linewidth=2)
ax.plot(fc_damped, label="Holt damped", color="tab:green", linestyle="--", linewidth=2)
ax.set_title("Linear Trend vs Damped Trend — US Population")
ax.set_ylabel("Population (thousands)")
ax.legend()
plt.tight_layout()
plt.show()

mae_damped = mean_absolute_error(pop_test["Population"], fc_damped)
print(f"MAE — Holt linear: {mae_holt:.2f}")
print(f"MAE — Holt damped: {mae_damped:.2f}")

---
## 7. Holt's Method on Airline Passengers

Airline passengers has both trend and seasonality.  Holt's method can
capture the trend but will miss the seasonal pattern.  This motivates
Holt-Winters (next notebook).

In [ ]:
# Linear trend
fit_air_lin = Holt(
    air_train["Passengers"],
    initialization_method="estimated",
).fit(optimized=True)

# Damped trend
fit_air_damp = Holt(
    air_train["Passengers"],
    damped_trend=True,
    initialization_method="estimated",
).fit(optimized=True)

fc_air_lin = fit_air_lin.forecast(len(air_test))
fc_air_damp = fit_air_damp.forecast(len(air_test))

print(f"Linear — alpha: {fit_air_lin.params['smoothing_level']:.4f}, "
      f"beta: {fit_air_lin.params['smoothing_trend']:.4f}")
print(f"Damped — alpha: {fit_air_damp.params['smoothing_level']:.4f}, "
      f"beta: {fit_air_damp.params['smoothing_trend']:.4f}, "
      f"phi: {fit_air_damp.params['damping_trend']:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(air_train["Passengers"], label="Train", color="black", alpha=0.5)
ax.plot(air_test["Passengers"], label="Actual (test)", color="tab:blue", linewidth=2)
ax.plot(fc_air_lin, label="Holt linear", color="tab:red", linestyle="--", linewidth=2)
ax.plot(fc_air_damp, label="Holt damped", color="tab:green", linestyle="--", linewidth=2)
ax.set_title("Holt's Method on Airline Passengers — Captures Trend, Misses Seasonality")
ax.set_ylabel("Thousands of Passengers")
ax.legend()
plt.tight_layout()
plt.show()

print("Holt's method captures the upward trend but produces a smooth line")
print("that completely misses the seasonal ups and downs.")
print("For trend + seasonality, we need Holt-Winters (next notebook).")

---
## 8. Long-Horizon Forecasts — Why Damping Matters

The danger of linear trend becomes apparent on long forecast horizons.
Let us forecast 10 years (120 months) ahead to see the divergence.

In [ ]:
LONG_HORIZON = 120

fc_long_lin = fit_air_lin.forecast(LONG_HORIZON)
fc_long_damp = fit_air_damp.forecast(LONG_HORIZON)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(airline["Passengers"], label="Actual data", color="tab:blue")
ax.plot(fc_long_lin, label="Linear (10 yr ahead)", color="tab:red", linestyle="--")
ax.plot(fc_long_damp, label="Damped (10 yr ahead)", color="tab:green", linestyle="--")
ax.set_title("Long-Horizon Forecasts: Linear Trend Diverges, Damped Flattens")
ax.set_ylabel("Thousands of Passengers")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Linear trend at h=120: {fc_long_lin.iloc[-1]:.0f}")
print(f"Damped trend at h=120: {fc_long_damp.iloc[-1]:.0f}")
print(f"\nThe linear forecast grows without bound — often unrealistic.")
print(f"The damped forecast flattens toward a finite asymptote.")

---
## 9. Fitted Values and Residuals

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# US population fitted values
axes[0].plot(pop_train["Population"], label="Actual", color="tab:blue")
axes[0].plot(fit_holt.fittedvalues, label="Fitted (linear)", color="tab:red",
             linestyle="--", alpha=0.8)
axes[0].plot(fit_damped.fittedvalues, label="Fitted (damped)", color="tab:green",
             linestyle="--", alpha=0.8)
axes[0].set_title("Fitted Values — US Population")
axes[0].set_ylabel("Population")
axes[0].legend(fontsize=9)

# Residuals
resid_lin = pop_train["Population"] - fit_holt.fittedvalues
resid_damp = pop_train["Population"] - fit_damped.fittedvalues
axes[1].plot(resid_lin, label="Linear residuals", alpha=0.7)
axes[1].plot(resid_damp, label="Damped residuals", alpha=0.7)
axes[1].axhline(0, color="black", linestyle="--")
axes[1].set_title("Residuals")
axes[1].set_ylabel("Residual")
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

---
## 10. Effect of Beta (Trend Smoothing Parameter)

- $\beta^*$ close to 0: the trend changes slowly (smooth trend).
- $\beta^*$ close to 1: the trend adapts quickly to recent slope changes.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(pop_train["Population"], color="black", alpha=0.3, label="Actual")

for beta in [0.01, 0.1, 0.5, 0.9]:
    m = Holt(pop_train["Population"], initialization_method="estimated")
    f = m.fit(smoothing_trend=beta, optimized=False)
    fc = f.forecast(len(pop_test))
    ax.plot(f.fittedvalues, alpha=0.6, label=f"beta={beta} (fitted)")
    ax.plot(fc, linestyle="--", alpha=0.8)

ax.plot(pop_test["Population"], color="tab:blue", linewidth=2, label="Actual (test)")
ax.axvline(pop_test.index[0], color="grey", linestyle=":", alpha=0.5)
ax.set_title("Effect of Beta* on Trend Smoothing")
ax.set_ylabel("Population")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

print("Low beta: smooth trend, less reactive to recent slope changes.")
print("High beta: volatile trend, tracks recent slope changes closely.")

---
## 11. Model Comparison Table

In [ ]:
# Compare SES, Holt linear, Holt damped on US population
from statsmodels.tsa.holtwinters import SimpleExpSmoothing

ses_fit = SimpleExpSmoothing(
    pop_train["Population"],
    initialization_method="estimated",
).fit(optimized=True)

fc_ses = ses_fit.forecast(len(pop_test))

actual_pop = pop_test["Population"].values

comparison = pd.DataFrame({
    "Method": ["SES", "Holt Linear", "Holt Damped"],
    "Alpha": [
        ses_fit.params["smoothing_level"],
        fit_holt.params["smoothing_level"],
        fit_damped.params["smoothing_level"],
    ],
    "Beta": [np.nan, fit_holt.params["smoothing_trend"], fit_damped.params["smoothing_trend"]],
    "Phi": [np.nan, np.nan, fit_damped.params["damping_trend"]],
    "AIC": [ses_fit.aic, fit_holt.aic, fit_damped.aic],
    "MAE": [
        mean_absolute_error(actual_pop, fc_ses),
        mean_absolute_error(actual_pop, fc_holt),
        mean_absolute_error(actual_pop, fc_damped),
    ],
})
comparison = comparison.round(4)
print(comparison.to_string(index=False))

print(f"\nBest by AIC: {comparison.loc[comparison['AIC'].idxmin(), 'Method']}")
print(f"Best by MAE: {comparison.loc[comparison['MAE'].idxmin(), 'Method']}")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(pop_train["Population"], label="Train", color="black", alpha=0.5)
ax.plot(pop_test["Population"], label="Actual (test)", color="tab:blue", linewidth=2)
ax.plot(fc_ses, label="SES (flat)", color="tab:orange", linestyle="--")
ax.plot(fc_holt, label="Holt linear", color="tab:red", linestyle="--")
ax.plot(fc_damped, label="Holt damped", color="tab:green", linestyle="--")
ax.set_title("SES vs Holt Linear vs Holt Damped — US Population")
ax.set_ylabel("Population (thousands)")
ax.legend()
plt.tight_layout()
plt.show()

---
## 12. The Level and Trend Components Visualized

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Level
axes[0].plot(pop_train.index, pop_train["Population"], color="black", alpha=0.4, label="Actual")
axes[0].plot(pop_train.index, fit_holt.level, color="tab:red", label="Level (linear)")
axes[0].plot(pop_train.index, fit_damped.level, color="tab:green", label="Level (damped)")
axes[0].set_title("Level Component")
axes[0].set_ylabel("Level")
axes[0].legend()

# Trend
axes[1].plot(pop_train.index, fit_holt.trend, color="tab:red", label="Trend (linear)")
axes[1].plot(pop_train.index, fit_damped.trend, color="tab:green", label="Trend (damped)")
axes[1].set_title("Trend Component (slope per period)")
axes[1].set_ylabel("Trend")
axes[1].legend()

plt.tight_layout()
plt.show()

---
## Summary

| Concept | Detail |
|---------|--------|
| **What** | SES + trend component — forecasts are straight lines |
| **Parameters** | $\alpha$ (level), $\beta^*$ (trend) |
| **Forecast shape** | Linear: $\ell_t + hb_t$ |
| **Damped trend** | Adds $\phi$ to prevent indefinite trend extrapolation |
| **Damped forecast** | $\ell_t + (\phi + \phi^2 + \cdots + \phi^h)b_t$ |
| **When $\phi = 1$** | Reduces to standard Holt's linear trend |
| **Typical $\phi$** | 0.80–0.98 |
| **Key insight** | Damped trend often wins in forecasting competitions |
| **Limitation** | Cannot capture seasonal patterns |

**Next notebook**: Holt-Winters — adding a seasonal component to handle
data with both trend and seasonality.

In [ ]:
print("Next notebook: Holt-Winters — adding seasonality to exponential smoothing.")